In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
import xml.etree.ElementTree as et

# **Data processing**

**XML Parsing**

In [4]:
sentiment_label_encode = {
    'NONE': 0,
    'P': 1,
    'N': 2,
    'NEU': 3
}

In [5]:
def xml_2_dataframe(path):
    xtree = et.parse(path)
    xroot = xtree.getroot()

    df = pd.DataFrame({
        'id': pd.Series(dtype='str'),
        'label': pd.Series(dtype='str'),
        'text': pd.Series(dtype='str')
    })

    for tweet in xroot:
        tweet_id = tweet.find('tweetid').text
        content = tweet.find('content').text
        sentiment = tweet.find('sentiment').find('polarity').find('value').text
        
        new_input = [tweet_id, sentiment, content]
        df.loc[len(df.index)] = new_input 

    return df

In [6]:
drive_path = 'drive/MyDrive/ALC/'

In [7]:
train_df = xml_2_dataframe(drive_path+'data/TASS2017_T1_training.xml')
dev_df = xml_2_dataframe(drive_path+'data/TASS2017_T1_development.xml')
test_df = xml_2_dataframe(drive_path+'data/TASS2017_T1_test.xml')

In [8]:
print(train_df.head())
print(dev_df.head())
print(test_df.head())

                   id label                                               text
0  768213876278165504  NONE  -Me caes muy bien \n-Tienes que jugar más part...
1  768213567418036224     N  @myendlesshazza a. que puto mal escribo\n\nb. ...
2  768212591105703936     N  @estherct209 jajajaja la tuya y la d mucha gen...
3  768221670255493120     P  Quiero mogollón a @AlbaBenito99 pero sobretodo...
4  768221021300264964     N  Vale he visto la tia bebiendose su regla y me ...
                   id label                                               text
0  770976639173951488     P  @noseashetero 1000/10 de verdad a ti que voy a...
1  771092421866389508     P  @piscolabisaereo @HistoriaNG @SPosteguillo las...
2  771092111429083136     P  Al final han sido 3h  Bueno, mañana tengo fies...
3  771092070572449796     N  @Jorge_Ruiz14 yo no tengo tiempo para esas cos...
4  771094192508600320     N  @_MissChaotic_ ves ese brillo? es un coso que ...
                   id label                         

**Tokenizer**

In [9]:
import re

def vocab_reducer(text):
    res = []
    for word in text.split():
        word = re.sub('@.*','mención', word)
        word = re.sub('#(.*)', 'etiqueta', word)
        word = re.sub('http.*', 'web', word)
        word = re.sub('\d.*', 'número', word)
        res.append(word)
    return ' '.join(res)


In [10]:
from nltk.tokenize import TweetTokenizer

def tokenize(dataframe):
    dataframe['tokenized_text'] = dataframe['text'].map(
        TweetTokenizer(strip_handles=False, reduce_len=True, preserve_case=False).tokenize
    )

    dataframe['tokenized_text'] = dataframe['tokenized_text'].map(
        ' '.join
    )

    dataframe['tokenized_text'] = dataframe['tokenized_text'].map(
        vocab_reducer
    )

    return dataframe

In [11]:
train_df = tokenize(train_df)
dev_df = tokenize(dev_df)
test_df = tokenize(test_df)

In [12]:
print(train_df.head)
print(dev_df.shape)
print(test_df.shape)

<bound method NDFrame.head of                       id label  \
0     768213876278165504  NONE   
1     768213567418036224     N   
2     768212591105703936     N   
3     768221670255493120     P   
4     768221021300264964     N   
...                  ...   ...   
1003  814846333601320960     P   
1004  813731371076243461     N   
1005  818399956792905728   NEU   
1006  815715581878009858     P   
1007  816978031357161476     P   

                                                   text  \
0     -Me caes muy bien \n-Tienes que jugar más part...   
1     @myendlesshazza a. que puto mal escribo\n\nb. ...   
2     @estherct209 jajajaja la tuya y la d mucha gen...   
3     Quiero mogollón a @AlbaBenito99 pero sobretodo...   
4     Vale he visto la tia bebiendose su regla y me ...   
...                                                 ...   
1003                  Para mí mi mejor año fue el 2015    
1004                  Hoy va a ser un dia muy largo...    
1005                     11:11

**Zero shot classification**

In [13]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 59.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.1/200.1 kB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 90.9 MB/s eta 0:00:00


In [14]:
from transformers import pipeline
roberta_zero_shot = pipeline('zero-shot-classification', model='Recognai/bert-base-spanish-wwm-cased-xnli', device=0)

In [15]:
labels = {
    'positivo': 'P',
    'negativo': 'N',
    'neutral': 'NEU',
    'indefinido': 'NONE' #'nada' : 'NONE'
  }


In [16]:
import numpy as np
def predict_sentiment(sentence):
  template='Este twit tiene sentimiento {}.'
  prediction = roberta_zero_shot(sentence, list(labels.keys()), hypothesis_template=template)
  winner_label_index = np.argmax(prediction['scores'])
  return labels[prediction['labels'][winner_label_index]]

In [17]:
train_dev_df = pd.concat([train_df, dev_df], ignore_index=True)

In [18]:
y_hat = [predict_sentiment(row['tokenized_text']) for _, row in train_dev_df.iterrows()]
y = train_dev_df['label']

In [19]:
from sklearn.metrics import classification_report
print(classification_report(y, y_hat))

              precision    recall  f1-score   support

           N       0.71      0.62      0.66       637
         NEU       0.18      0.37      0.24       202
        NONE       0.26      0.07      0.12       201
           P       0.64      0.64      0.64       474

    accuracy                           0.52      1514
   macro avg       0.45      0.43      0.41      1514
weighted avg       0.55      0.52      0.53      1514



**Model fine-tuning**

In [20]:
!pip install datasets transformers[sentencepiece]

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.7/468.7 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 kB 27.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 18.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 24.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstallin

In [21]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import DataCollatorWithPadding, AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer, AutoModel, AutoConfig
import torch
import torch.nn as nn
import pandas as pd

In [22]:
def label_parser(label):
  l = ['P', 'N', 'NEU', 'NONE']
  l_id = l.index(label) if label in l else 0
  return l_id

In [23]:
def parse_dataset_label(dataset):
  dataset['dec_label'] = dataset['label']
  dataset['label'] = dataset['label'].map(
    label_parser
  )
  return dataset

In [24]:
train_df = parse_dataset_label(dataset=train_df)
dev_df = parse_dataset_label(dataset=dev_df)
test_df = parse_dataset_label(dataset=test_df)
dev_df.head()

,id,label,text,tokenized_text,dec_label
0,770976639173951488,0,@noseashetero 1000/10 de verdad a ti que voy a...,mención número de verdad a ti que voy a decir ...,P
1,771092421866389508,0,@piscolabisaereo @HistoriaNG @SPosteguillo las...,mención mención mención las tengo pero aún no ...,P
2,771092111429083136,0,"Al final han sido 3h Bueno, mañana tengo fies...","al final han sido número bueno , mañana tengo ...",P
3,771092070572449796,1,@Jorge_Ruiz14 yo no tengo tiempo para esas cos...,mención yo no tengo tiempo para esas cosas aho...,N
4,771094192508600320,1,@_MissChaotic_ ves ese brillo? es un coso que ...,mención ves ese brillo ? es un coso que hace q...,N


In [25]:
train_dev_df = pd.concat([train_df, dev_df], ignore_index=True)

train_ds = Dataset.from_pandas(train_df)
eval_ds = Dataset.from_pandas(dev_df)
test_ds = Dataset.from_pandas(test_df)
#eval_ds = eval_ds.train_test_split(test_size=0.2)
dataset = DatasetDict({
    'train': train_ds,
    'valid': eval_ds,#eval_ds['train'],
    'test': test_ds#eval_ds['test']
})

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__'],
        num_rows: 1008
    })
    valid: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__'],
        num_rows: 506
    })
    test: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__'],
        num_rows: 1899
    })
})

In [26]:
model_name = 'dccuchile/bert-base-spanish-wwm-cased'

In [27]:
roberta_tok = AutoTokenizer.from_pretrained(model_name)
roberta_tok.model_max_len = 512

In [28]:
def tokenize(batch):
  return roberta_tok(batch['text'], truncation=True, max_length=512)

tok_dataset = dataset.map(
    tokenize,
    batched=True
)
tok_dataset

Map:   0%|          | 0/1008 [00:00<?, ? examples/s]

Map:   0%|          | 0/506 [00:00<?, ? examples/s]

Map:   0%|          | 0/1899 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1008
    })
    valid: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 506
    })
    test: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1899
    })
})

In [29]:
tok_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
data_collator = DataCollatorWithPadding(tokenizer=roberta_tok)
tok_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1008
    })
    valid: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 506
    })
    test: Dataset({
        features: ['id', 'label', 'text', 'tokenized_text', 'dec_label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1899
    })
})

In [30]:
from transformers.modeling_outputs import TokenClassifierOutput
class SentimentAnalisysModel(nn.Module):
  def __init__(self, model_name):
    super(SentimentAnalisysModel, self).__init__()
    self.num_labels = 4

    self.model = AutoModel.from_pretrained(
        model_name, 
        config=AutoConfig.from_pretrained(
            model_name, 
            output_attentions=True, 
            output_hidden_states=True
            )
        )
    
    self.dropout = nn.Dropout(0.1)
    self.classifier = nn.Linear(768, self.num_labels)

  def forward(self, input_ids=None, attention_mask=None, labels=None):
    outs = self.model(input_ids=input_ids, attention_mask=attention_mask)

    last_hidden_state = self.dropout(outs[0])

    logits = self.classifier(last_hidden_state[:,0,:].view(-1,768))

    loss = None
    if labels != None:
      loss_fn = nn.CrossEntropyLoss()
      loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

    return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outs.hidden_states, attentions=outs.attentions)

In [31]:
device = torch.device('cuda')
model=SentimentAnalisysModel(model_name=model_name).to(device)

Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [32]:
from torch.utils.data import DataLoader

tr_dataloader = DataLoader(
    tok_dataset['train'], shuffle=True, batch_size=16, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tok_dataset['valid'], shuffle=True, batch_size=16, collate_fn=data_collator
)

In [33]:
from transformers import AdamW, get_scheduler

opt = AdamW(model.parameters(), lr=5e-5)

epochs=10
tr_steps = epochs * len(tr_dataloader)

lr_scheduler = get_scheduler(
    'linear',
    optimizer=opt,
    num_warmup_steps=0,
    num_training_steps=tr_steps
)

In [34]:
from datasets import load_metric, list_metrics
metric = load_metric("f1")

In [35]:
from tqdm.auto import tqdm

prog_bar_tr = tqdm(range(tr_steps))
prog_bar_eval = tqdm(range(epochs * len(eval_dataloader)))

for epoch in range(epochs):
  model.train()
  for batch in tr_dataloader:
    batch = {k: v.to(device) for k,v in batch.items()}
    outs = model(**batch)
    loss = outs.loss
    loss.backward()

    opt.step()
    lr_scheduler.step()
    opt.zero_grad()
    prog_bar_tr.update(1)

  model.eval()
  for batch in eval_dataloader:
    batch = {k: v.to(device) for k,v in batch.items()}
    with torch.no_grad():
      outs = model(**batch)

    logits = outs.logits
    preds = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=preds, references=batch['labels'])
    prog_bar_eval.update(1)

  print(metric.compute(average='macro'))


  0%|          | 0/630 [00:00<?, ?it/s]

  0%|          | 0/320 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'f1': 0.4728666847000562}
{'f1': 0.5206647244191346}
{'f1': 0.5493103595375954}
{'f1': 0.5412810232019205}
{'f1': 0.5398889445679178}
{'f1': 0.5321444827294581}
{'f1': 0.5258062921273182}
{'f1': 0.5140355754850154}
{'f1': 0.5185054050483268}
{'f1': 0.5245412146051919}


In [36]:
model.eval()

test_dataloader = DataLoader(
    tok_dataset['valid'], collate_fn=data_collator, shuffle=False
)
preds = []
y = []
for batch in test_dataloader:
  batch = {k:v.to(device) for k,v in batch.items()}
  print(batch)
  with torch.no_grad():
    outs = model(**batch)

  logits = outs.logits
  pred = torch.argmax(logits, dim=-1)
  preds.append(pred)
  y.append(batch['labels'])
  metric.add_batch(predictions=pred, references=batch['labels'])

metric.compute(average='macro')

{'input_ids': tensor([[    4,   968,  1445,  3417,  1874,  3883, 12599,   972,  1681,  1008,
          1836,  1013,  1248,  1038,  2113,  1013,  1631,  1407,  1220,  1093,
          1038,  1240,  1937,  1216,  1038,  1013,  1857,  1698,     3,     5]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='cuda:0'), 'labels': tensor([0], device='cuda:0')}
{'input_ids': tensor([[    4,   968,  6656,  4866, 27432,  1889, 19353,   968, 10245, 30969,
         30992,   968, 14614, 25008, 24447,  1085,  1542,  1089,  1847,  1355,
          2663,  1084,  1089,  1734, 11596,  1009,  1906,  1015,  1176,  5691,
         11837,     5]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0'), 'labels': tensor([0], device='cuda:0')}
{'input_ids': tensor([[    4,  1403,  2483,  1600,

{'f1': 0.5245412146051919}

In [37]:
preds = torch.Tensor(preds).tolist()
y = torch.Tensor(y).tolist()

from sklearn.metrics import classification_report
print(classification_report(y, preds))

              precision    recall  f1-score   support

         0.0       0.73      0.72      0.72       156
         1.0       0.73      0.73      0.73       219
         2.0       0.22      0.29      0.25        69
         3.0       0.48      0.34      0.40        62

    accuracy                           0.62       506
   macro avg       0.54      0.52      0.52       506
weighted avg       0.63      0.62      0.62       506



In [38]:
model.eval()

test_dataloader = DataLoader(
    tok_dataset['test'], collate_fn=data_collator, shuffle=False
)
preds = []
for batch in test_dataloader:
  batch = {k:v.to(device) for k,v in batch.items()}
  with torch.no_grad():
    outs = model(**batch)

  logits = outs.logits
  pred = torch.argmax(logits, dim=-1)
  preds.append(pred)

preds = torch.Tensor(preds).tolist()

labels = ['P', 'N', 'NEU', 'NONE']

with open('drive/MyDrive/ALC/MiguelGarcia_BETO.txt','w') as file:
  for i, label in enumerate(preds):
    file.write('{}\t{}\n'.format(test_df['id'][i], labels[int(preds[i])]))